In [11]:
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
import random
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

from improved_model import (
    build_all_models, 
    TemporalConsistencyLoss, 
    LabelSmoothingLoss
)

In [12]:
# ========================= CONFIG =========================
DATA_DIR = "dataset_word/landmarks/final"
ABLATION_DIR = "ablation_results"
MODEL_DIR = "ablation_models"
os.makedirs(ABLATION_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [ ]:
# ========================= FEATURE MODES (shared) =========================
import sys
sys.path.insert(0, 'scripts')
from ablation_study_dynamic import load_data

# Note: `load_data` now lives in `scripts/ablation_study_dynamic.py` to avoid duplication

In [14]:

# ===================== MAIN ABLATION FUNCTION =====================
def run_ablation(experiment_name, 
                 feature_mode="full",
                 model_type="BiGRU_Attention",
                 use_temporal_loss=True,
                 seq_length=40,
                 epochs=60,
                 note=""):
    
    print(f"\n{'='*90}")
    print(f"STARTING ABLATION: {experiment_name}")
    print(f"Feature: {feature_mode} | Model: {model_type} | Temporal Loss: {use_temporal_loss} | Seq Len: {seq_length}")
    print(f"{'='*90}")
    
    # Load data
    X, y, class_names = load_data(feature_mode, target_length=seq_length)
    
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    
    # Train/Val/Test Split
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=SEED)
    
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=SEED)
    
    input_shape = (seq_length, X.shape[2])
    
    # Build Model
    models_dict = build_all_models(input_shape, num_classes=len(class_names))
    model = models_dict[model_type]
    
    # Loss
    if use_temporal_loss:
        loss_fn = TemporalConsistencyLoss(
            num_classes=len(class_names), 
            label_smoothing=0.1, 
            temporal_weight=0.05
        )
    else:
        loss_fn = LabelSmoothingLoss(num_classes=len(class_names), smoothing=0.1)
    
    # Optimizer
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001, clipnorm=1.0)
    
    model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])
    
    # Callbacks
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=12, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(MODEL_DIR, f"{experiment_name}_best.h5"),
            monitor='val_accuracy', save_best_only=True, verbose=0)
    ]
    
    # Training
    start = datetime.now()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=16,
        callbacks=callbacks,
        verbose=1
    )
    train_time = (datetime.now() - start).total_seconds()
    
    # Evaluation
    val_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)
    test_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    
    val_acc = accuracy_score(y_val, val_pred)
    test_acc = accuracy_score(y_test, test_pred)
    
    # Save result
    result = {
        "experiment": experiment_name,
        "feature_mode": feature_mode,
        "model_type": model_type,
        "use_temporal_loss": use_temporal_loss,
        "seq_length": seq_length,
        "val_accuracy": float(val_acc),
        "test_accuracy": float(test_acc),
        "training_time_sec": train_time,
        "note": note,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M")
    }
    
    # Save to CSV
    csv_path = os.path.join(ABLATION_DIR, "ablation_results.csv")
    pd.DataFrame([result]).to_csv(csv_path, mode='a', header=not os.path.exists(csv_path), index=False)
    
    print(f"✅ FINISHED → Val: {val_acc:.4f} | Test: {test_acc:.4f} | Time: {train_time:.1f}s\n")
    return result
# ========================= RUN ABLATIONS =========================
if __name__ == "__main__":
    
    experiments = [
        # 1. Feature Ablation
        ("01_Full_Features",          "full",             "BiGRU_Attention", True, 40),
        ("02_Landmarks_Only",         "landmarks_only",   "BiGRU_Attention", True, 40),
        ("03_Landmarks_Vel",          "landmarks_vel",    "BiGRU_Attention", True, 40),
        ("04_No_Geometric",           "no_geometric",     "BiGRU_Attention", True, 40),
        ("05_No_Bones",               "no_bones",         "BiGRU_Attention", True, 40),
        
        # 2. Model Architecture Ablation
        ("06_GRU_Baseline",           "full",             "GRU",             True, 40),
        ("07_BiGRU",                  "full",             "BiGRU",           True, 40),
        ("08_BiGRU_Attention",        "full",             "BiGRU_Attention", True, 40),
        
        # 3. Loss Ablation
        ("09_No_Temporal_Loss",       "full",             "BiGRU_Attention", False, 40),
        
        # 4. Sequence Length
        ("10_SeqLen_20",              "full",             "BiGRU_Attention", True, 20),
        ("11_SeqLen_30",              "full",             "BiGRU_Attention", True, 30),
        ("12_SeqLen_60",              "full",             "BiGRU_Attention", True, 60),
    ]
    
    print("🚀 Starting Ablation Study...\n")
    
    for exp in experiments:
        run_ablation(
            experiment_name=exp[0],
            feature_mode=exp[1],
            model_type=exp[2],
            use_temporal_loss=exp[3],
            seq_length=exp[4],
            epochs=65,           # Reduced for faster ablation
            note="Ablation study"
        )
    
    print("\n🎉 ALL ABLATION EXPERIMENTS COMPLETED!")
    print(f"Results saved to: {ABLATION_DIR}/ablation_results.csv")

🚀 Starting Ablation Study...


STARTING ABLATION: 01_Full_Features
Feature: full | Model: BiGRU_Attention | Temporal Loss: True | Seq Len: 40
Loading data with mode: full


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'dataset_word/landmarks/final'

🚀 Starting Ablation Study...


STARTING ABLATION: 01_Full_Features
Feature: full | Model: BiGRU_Attention | Temporal Loss: True | Seq Len: 40
Loading data with mode: full


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'dataset_word/landmarks/final'